In [ ]:
import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)
from kafi.streams.topologynode import TopologyNode as Tn

#

order_source_str = "orders"
order_source_tn = Tn.source(order_source_str)

cheap_order_tn = order_source_tn.filter(lambda x: x["amount"] < 10)
expensive_order_tn = order_source_tn.filter(lambda x: x["amount"] >= 10)

#

cheap_order_tn.build()
expensive_order_tn.build()

#

order_source_tn.to_zSet(Tn._from_records)
cheap_order_tn.from_zSet(Tn._to_records)
expensive_order_tn.from_zSet(Tn._to_records)

#

def gen_order(order_id_int, amount_int):
    return {"order_id": order_id_int, "amount": amount_int}
#

cheap_order_tn.push(order_source_str, [(gen_order(1, 1), 1), (gen_order(1, 10), 1)])
expensive_order_tn.push(order_source_str, [(gen_order(1, 1), 1), (gen_order(1, 10), 1)])
print(cheap_order_tn.latest())
print(expensive_order_tn.latest())


Step 1
[({'order_id': 1, 'amount': 1}, 1)]
[({'order_id': 1, 'amount': 10}, 1)]


In [51]:
import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)
from kafi.streams.topologynode import TopologyNode as Tn

#

order_source_str = "orders"
order_source_tn = Tn.source(order_source_str)

cheap_order_tn = order_source_tn.filter(lambda x: x["amount"] > 0 and x["amount"] < 10)
expensive_order_tn = order_source_tn.filter(lambda x: x["amount"] >= 10)
zero_order_tn = order_source_tn.filter(lambda x: x["amount"] == 0)

root_tn = Tn.sink(("cheap", cheap_order_tn), ("expensive", expensive_order_tn), ("zero", zero_order_tn))

#

# print(root_tn.topology())

root_tn.build()

#

order_source_tn.to_zSet(Tn._from_records)
cheap_order_tn.from_zSet(Tn._to_records)
zero_order_tn.from_zSet(Tn._to_records)

root_tn.from_zSet(Tn._to_records)

#

def gen_order(order_id_int, amount_int):
    return {"order_id": order_id_int, "amount": amount_int}
#
root_tn._sink_str_list = None
root_tn.push(order_source_str, [(gen_order(1, 1), 1), (gen_order(2, 10), 1), (gen_order(3, 0), 1)])
print(root_tn.latest())
root_tn.push(order_source_str, [(gen_order(1, 1), -1), (gen_order(4, 10), 1)])
print(root_tn.latest())


[(['cheap', {'order_id': 1, 'amount': 1}], 1), (['expensive', {'order_id': 2, 'amount': 10}], 1), (['zero', {'order_id': 3, 'amount': 0}], 1)]
[(['cheap', {'order_id': 1, 'amount': 1}], -1), (['expensive', {'order_id': 4, 'amount': 10}], 1)]
